# Masked DFM on K562: minimal train + sample
Runtime → Change runtime type → GPU.

In [ ]:
!git clone -b experimental https://github.com/darreld081/CIS6270-Final-Project.git
%cd CIS6270-Final-Project
!pip install -q pytorch-lightning
import sys; sys.path.insert(0, "src")

In [2]:
%cd CIS6270-Final-Project
import sys; sys.path.insert(0, "src")

[Errno 2] No such file or directory: 'CIS6270-Final-Project'
/home/darre/CIS6270-Final-Project


In [ ]:
# Preprocess: download raw DHS table, keep K562, encode to int8[N, 200]
from dfm.data import download, load_table, build_splits

splits = build_splits(load_table(download("data/raw")))
splits["train"].shape

In [4]:
# Train
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from dfm.module import MaskedDFM

EPOCHS = 200

pl.seed_everything(0)
train_loader = DataLoader(torch.from_numpy(splits["train"]).long(),
                          batch_size=256, shuffle=True, drop_last=True, num_workers=2)
module = MaskedDFM()
trainer = pl.Trainer(max_epochs=EPOCHS, gradient_clip_val=1.0,
                     precision="bf16-mixed" if torch.cuda.is_available() else "32-true",
                     logger=False, enable_checkpointing=False)
trainer.fit(module, train_loader)

Seed set to 0
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/darre/CIS6270-Final-Project/.venv/lib/python3.12/site-packages/pytorch_lightning/trainer/configuration_validator.py:70: You defined a `validation_step` but have no `val_dataloader`. Skipping val loop.
Loading `train_dataloader` to estimate number of stepping batches.
/home/darre/CIS6270-Final-Project/.venv/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

  | Name  | Type        | Params | Mode  | FLOPs
------------------------------------------------------
0 | model | CNNDenoiser | 4.0 M  | train | 0    
------------------------

Epoch 3:  45%|████▍     | 17/38 [01:45<02:10,  0.16it/s, train_loss_step=1.300, train_loss_epoch=1.320]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/darre/CIS6270-Final-Project/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [5]:
# Sample one sequence
from dfm.alphabet import decode
from dfm.sampler import sample

device = "cuda" if torch.cuda.is_available() else "cpu"
model = module.model.to(device).eval()
seq = sample(model, num_samples=1, num_steps=128, device=device, seed=0)
print(decode(seq[0]))

CAGAAGGCAGCAGTCGACCACTTTTCACCTGAATGACAGAGAGTCATTACTGCCATCAGTTCAGAAGAGGAAAAGAGAACGTCGAGGGTCATGGGGTGGGGGTTGAGGACAGAGCCGGCTGTGGGTGAGGGCGCATGGGCCTCAGAGGGCGTGCCTTGGGCAGCTGGGCAGAGCCCGGTGGGGGGGAGGAGATGCCCATA


In [ ]:
# Evaluate 1024 samples against held-out test sequences (chr1).
# Good: k-mer JSD well below the uniform baseline, GC mean close to test and std ratio near 1,
# unique near 1, memorized near 0, pairwise Hamming close to test (much lower = mode collapse).
from dfm.metrics import evaluate

gen = sample(model, num_samples=1024, num_steps=128, batch_size=256, device=device, seed=1)
r = evaluate(gen, splits["test"], train=splits["train"])
print(f"GC mean   gen {r['gc_generated_mean']:.3f}  test {r['gc_reference_mean']:.3f}  std ratio {r['gc_std_ratio']:.2f}")
for k in r["kmer_jsd"]:
    print(f"{k}-mer JSD  gen {r['kmer_jsd'][k]:.4f}  uniform {r['kmer_jsd_uniform_baseline'][k]:.4f}")
print(f"unique {r['unique_fraction']:.3f}  memorized {r['memorization_fraction']:.3f}")
print(f"pairwise Hamming  gen {r['mean_pairwise_hamming']:.3f}  test {r['reference_pairwise_hamming']:.3f}")